# microGPT on PYNQ-Z2 -- demo

This notebook loads the `microgpt.bit` overlay onto the Zynq PL and
drives the AXI4-Lite slave through the `MicroGPT` Python class to
generate five short names, printing cycle counts for each run.

Before running, copy `overlays/microgpt.bit` and `overlays/microgpt.hwh`
to the board (or run this notebook directly on the board).


In [1]:
import sys
from pathlib import Path

# Make the driver importable regardless of where the notebook is launched.
drivers_path = Path('..') / 'drivers'
if str(drivers_path.resolve()) not in sys.path:
    sys.path.insert(0, str(drivers_path.resolve()))

from microgpt import MicroGPT


ModuleNotFoundError: No module named 'pynq'

In [ ]:
gpt = MicroGPT()
print('Loaded overlay; status:', gpt.status())


## 5 generations

Each row prints the seed, the generated name, the cycle count
from the core's performance counter, and the final status word.

In [ ]:
seeds = [1, 42, 1337, 0xDEADBEEF, 0xC0FFEE]
for s in seeds:
    text, info = gpt.generate(max_tokens=8, temperature=1.0, seed=s)
    print(f"seed=0x{s:08X}  text={text!r:12s}  cycles={info['cycles']:>8d}  tokens={info['tokens']}")


## Single-step direct-mode probe

This pokes a single token+position into the core and reads back
the 27 logits the lm_head produced. Useful for debugging.

In [ ]:
result = gpt.step(token=26, pos=0, clear=True, seed=1)
print('argmax token:', result['argmax_token'])
print('top logit (Q12):', result['top_logit_q12'])
print('all logits:', gpt.logits())
